# Lesson 4: Persistence and Streaming

In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
import os
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SerpAPIWrapper

In [3]:
search = SerpAPIWrapper(serpapi_api_key=os.getenv("SERP_API_KEY"))

@tool
def serpapi_search(query: str) -> str:
    """Search the web with SerpAPI. Use for current information, news, weather."""
    return search.run(query)

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [5]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect(":memory:", check_same_thread=False)
memory = SqliteSaver(conn)

In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [7]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [serpapi_search], system=prompt, checkpointer=memory)

In [8]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [9]:
thread = {"configurable": {"thread_id": "1"}}

In [10]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_lobeRWbJBMuy52qw0bxbiqJG', 'function': {'arguments': '{"query":"San Francisco weather"}', 'name': 'serpapi_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 127, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_d00f95a9ec', 'id': 'chatcmpl-EnRZw6c0i8LTaQoTPxsWlk45ub8Bs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--d2d8d04f-e714-4be1-ba78-db13ab79a4ca-0', tool_calls=[{'name': 'serpapi_search', 'args': {'query': 'San Francisco weather'}, 'id': 'call_lobeRWbJBMuy52qw0bxbiqJG', 'type': 'tool_call'}], usage_metadata={'input_tokens': 127, 'output_tokens': 18, 'total_token

In [11]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_ED26UcgtUGTqeOItpQnenxJc', 'function': {'arguments': '{"query":"Los Angeles weather"}', 'name': 'serpapi_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 1585, 'total_tokens': 1603, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1408}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_d00f95a9ec', 'id': 'chatcmpl-EnRa0ppQRRHWbAGbI6wi03VxZxvN6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--fc452b5c-42c7-460f-821a-58644292e99b-0', tool_calls=[{'name': 'serpapi_search', 'args': {'query': 'Los Angeles weather'}, 'id': 'call_ED26UcgtUGTqeOItpQnenxJc', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1585, 'output_tokens': 1

In [12]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Los Angeles is warmer compared to San Francisco. The temperature in Los Angeles is 59°F, while in San Francisco, it is 47°F.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 2494, 'total_tokens': 2525, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 2432}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_d00f95a9ec', 'id': 'chatcmpl-EnRa38FaMsmfUDQjvDMD4zUbKP2UF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--b8c5c427-57b3-43b1-887f-6f3a968796eb-0', usage_metadata={'input_tokens': 2494, 'output_tokens': 31, 'total_tokens': 2525, 'input_token_details': {'audio': 0, 'cache_read': 2432}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}


In [13]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='Could you please provide me with the specific locations or regions you are comparing in terms of temperature?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 125, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_d00f95a9ec', 'id': 'chatcmpl-EnRa5jvXSWwEanIHX9wZQkHKbwsnh', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--fee97245-c06c-40c3-b799-d56473602b52-0', usage_metadata={'input_tokens': 125, 'output_tokens': 20, 'total_tokens': 145, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}


## Streaming tokens

In [14]:
import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

conn = await aiosqlite.connect(":memory:")
memory = AsyncSqliteSaver(conn)
abot = Agent(model, [serpapi_search], system=prompt, checkpointer=memory)

In [15]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

Calling: {'name': 'serpapi_search', 'args': {'query': 'current weather in San Francisco'}, 'id': 'call_4G7m0zfnqLKaClkVvtkYxF81', 'type': 'tool_call'}
Back to the model!
The| current| weather| in| San| Francisco|,| CA| is| mostly| cloudy| with| a| temperature| of| |47|°F|.| The| humidity| is| |91|%,| and| there| is| a| |10|%| chance| of| precipitation|.| The| wind| is| blowing| at| |7| mph|.|